# Full Fine-tuning বনাম Parameter-Efficient Fine-tuning

একটি memory-footprint calculator (সাধারণ পাটিগণিত, কোনো model আসলে instantiate করা হয় না — মূল বিষয় হলো **ORDER OF MAGNITUDE**, যা বাস্তব framework-গুলো হুবহু নিশ্চিত করে)। তিনটি demo:

1. AdamW দিয়ে full fine-tuning-এর প্রতি-parameter memory খরচ (weights + gradients + fp32 master copy + Adam-এর দুটি moment) বনাম PEFT (frozen base, যার খরচ প্রতি parameter মাত্র 2 bytes, সাথে ছোট একটি trainable অংশ যার খরচ পুরো training হারে)।
2. সেই তুলনাটিই বাস্তবসম্মত model size-গুলোর ওপর sweep করা — 125M থেকে 70B parameter পর্যন্ত।
3. Multi-task deployment-এর দিক: N-টি full fine-tuned checkpoint সংরক্ষণ বনাম একটি shared frozen base সাথে N-টি ছোট adapter।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। এখানে কোনো import বা model লাগে না — সবই সাধারণ পাটিগণিত, তাই মুহূর্তেই চলে।

## 0. Constants: mixed-precision AdamW memory recipe

README-এর section 2-এর memory recipe-টি এখানে constant হিসেবে সংজ্ঞায়িত: একটি trainable parameter-এর জন্য মোট 16 bytes, আর একটি frozen parameter-এর জন্য মাত্র 2 bytes। সাথে দুটি helper function — `full_finetune_bytes` ও `peft_bytes` — যা পরের সব demo ব্যবহার করে।

In [ ]:
# ---------------------------------------------------------------------------
# Constants: mixed-precision AdamW memory recipe (README section 2)
# ---------------------------------------------------------------------------

BYTES_WEIGHTS_FP16 = 2
BYTES_GRADIENTS_FP16 = 2
BYTES_FP32_MASTER_COPY = 4
BYTES_ADAM_MOMENT_1 = 4
BYTES_ADAM_MOMENT_2 = 4

# Optimization চলাকালীন একটি TRAINABLE parameter-এর যা যা লাগে।
BYTES_PER_TRAINABLE_PARAM = (
    BYTES_WEIGHTS_FP16
    + BYTES_GRADIENTS_FP16
    + BYTES_FP32_MASTER_COPY
    + BYTES_ADAM_MOMENT_1
    + BYTES_ADAM_MOMENT_2
)  # 16 bytes/param

# একটি FROZEN parameter-কে শুধু forward pass-এর জন্য memory-তে থাকতে হয় --
# কোনো gradient নেই, কোনো optimizer state নেই, কোনো fp32 master copy নেই।
BYTES_PER_FROZEN_PARAM = BYTES_WEIGHTS_FP16  # 2 bytes/param

GB = 1024 ** 3


def full_finetune_bytes(n_total):
    return n_total * BYTES_PER_TRAINABLE_PARAM


def peft_bytes(n_total, n_trainable):
    frozen = n_total * BYTES_PER_FROZEN_PARAM
    trainable = n_trainable * BYTES_PER_TRAINABLE_PARAM
    return frozen + trainable

## 1. প্রতি-parameter খরচের বিস্তারিত হিসাব

AdamW training-এর সময় একটি trainable parameter-এর খরচ কোন কোন অংশ দিয়ে তৈরি — weights, gradients, fp32 master copy আর Adam-এর দুটি moment — সেটি এক এক করে print করা হয়, এবং frozen parameter-এর 2 bytes-এর সাথে তুলনা করে 8x পার্থক্যটি দেখানো হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. প্রতি-parameter খরচের বিস্তারিত হিসাব, এক এক করে লেখা
# ---------------------------------------------------------------------------

def per_parameter_breakdown_demo():
    print("=" * 78)
    print("1. WHAT ONE TRAINABLE PARAMETER COSTS DURING ADAMW TRAINING")
    print("=" * 78)
    print(f"  weights (fp16)                   {BYTES_WEIGHTS_FP16} bytes")
    print(f"  gradients (fp16)                 {BYTES_GRADIENTS_FP16} bytes")
    print(f"  fp32 master weight copy          {BYTES_FP32_MASTER_COPY} bytes")
    print(f"  Adam moment 1 (fp32)             {BYTES_ADAM_MOMENT_1} bytes")
    print(f"  Adam moment 2 (fp32)             {BYTES_ADAM_MOMENT_2} bytes")
    print(f"  {'-' * 40}")
    print(f"  TOTAL per trainable parameter    {BYTES_PER_TRAINABLE_PARAM} bytes")
    print(f"\n  A FROZEN parameter (PEFT's base model) only pays for inference")
    print(f"  storage: {BYTES_PER_FROZEN_PARAM} bytes/param -- a "
          f"{BYTES_PER_TRAINABLE_PARAM / BYTES_PER_FROZEN_PARAM:.0f}x difference,")
    print("  per parameter, between 'frozen' and 'trainable under AdamW'.")


per_parameter_breakdown_demo()

## 2. বাস্তবসম্মত model size-গুলোর ওপর sweep

GPT-2 small থেকে Llama-2-70B পর্যন্ত বিভিন্ন model size-এর জন্য full fine-tuning ও PEFT-এর মোট training memory পাশাপাশি হিসাব করা হয় (PEFT ধরে নেয় 0.5% parameter trainable)। তারপর শুধু gradient + optimizer-state overhead আলাদা করে দেখানো হয় — যেখানে পার্থক্যটি প্রতিটি size-এ হুবহু 1/fraction = 200x, অর্থাৎ সত্যিকারের কয়েক order of magnitude।

In [ ]:
# ---------------------------------------------------------------------------
# 2. বাস্তবসম্মত model size-গুলোর ওপর sweep
# ---------------------------------------------------------------------------

MODEL_SIZES = [
    ("GPT-2 small-scale", 125e6),
    ("GPT-2 XL-scale", 1.5e9),
    ("GPT-3-Curie-scale", 6.7e9),
    ("Llama-2-7B-scale", 7e9),
    ("Llama-2-13B-scale", 13e9),
    ("Llama-2-70B-scale", 70e9),
]

# LoRA-ধাঁচের PEFT-এর জন্য একটি প্রতিনিধিত্বমূলক trainable fraction (Lesson 2
# একটি নির্দিষ্ট config-এর জন্য হুবহু সংখ্যাটি হিসাব করে; সাধারণ rank/target-module
# পছন্দের ক্ষেত্রে LoRA/QLoRA paper-গুলোতে উদ্ধৃত বাস্তবসম্মত পরিসর ~0.1%-1%)।
PEFT_TRAINABLE_FRACTION = 0.005  # মোট parameter-এর 0.5% trainable


def model_size_sweep_demo():
    print("\n" + "=" * 78)
    print("2. TRAINING MEMORY: FULL FINE-TUNING vs. PEFT, ACROSS MODEL SIZES")
    print("=" * 78)
    print(f"(PEFT assumes a {PEFT_TRAINABLE_FRACTION * 100:.1f}% trainable-parameter "
          f"fraction, in line with typical LoRA configs -- see Lesson 2)\n")

    header = (f"{'model':<20}{'total params':>14}{'full FT (GB)':>15}"
              f"{'PEFT (GB)':>12}{'ratio':>10}")
    print(header)
    for name, n_total in MODEL_SIZES:
        n_trainable = n_total * PEFT_TRAINABLE_FRACTION
        full_gb = full_finetune_bytes(n_total) / GB
        peft_gb = peft_bytes(n_total, n_trainable) / GB
        ratio = full_gb / peft_gb
        print(f"{name:<20}{n_total:>14.2e}{full_gb:>15.1f}{peft_gb:>12.2f}{ratio:>9.1f}x")

    llama7b_full = full_finetune_bytes(7e9) / GB
    llama7b_peft = peft_bytes(7e9, 7e9 * PEFT_TRAINABLE_FRACTION) / GB
    llama7b_inference_only = 7e9 * BYTES_PER_FROZEN_PARAM / GB
    print(f"\n-> Full fine-tuning a 7B model needs about {llama7b_full:.0f} GB -- more than")
    print(f"   an 80 GB A100 has by itself. Just LOADING that same model for inference")
    print(f"   (no training at all) needs only about {llama7b_inference_only:.0f} GB. PEFT training")
    print(f"   sits at about {llama7b_peft:.1f} GB -- barely above the inference-only cost --")
    print(f"   because {(1 - PEFT_TRAINABLE_FRACTION) * 100:.1f}% of the model never needs a gradient,")
    print("   an optimizer state, or an fp32 master copy at all.")

    # ওপরের মোট-memory ratio-টি প্রায় 8x-এ গিয়ে থেমে যায়, কারণ forward pass
    # চালাতেই দুই পদ্ধতিকেই base model-এর weights সংরক্ষণ করতে হয় -- এই shared
    # খরচ ব্যবধানটিকে পাতলা করে দেয়। যে ব্যবধানটি সত্যিই কয়েক order of magnitude,
    # সেটি হলো সেই অংশে যা শুধু TRAINABLE parameter-এর সাথে scale করে:
    # gradients + fp32 master copy + Adam-এর দুটি moment (16-byte হিসাবের সবকিছু,
    # শুধু 2-byte weights বাদে)।
    overhead_per_trainable_param = BYTES_PER_TRAINABLE_PARAM - BYTES_WEIGHTS_FP16  # 14 bytes
    print(f"\n{'':30s}{'trainable params':>18}{'grad+optimizer overhead (GB)':>30}")
    for name, n_total in MODEL_SIZES:
        n_trainable_full = n_total
        n_trainable_peft = n_total * PEFT_TRAINABLE_FRACTION
        full_overhead_gb = n_trainable_full * overhead_per_trainable_param / GB
        peft_overhead_gb = n_trainable_peft * overhead_per_trainable_param / GB
        print(f"{name + ' (full FT)':<30s}{n_trainable_full:>18.2e}{full_overhead_gb:>30.1f}")
        print(f"{name + ' (PEFT)':<30s}{n_trainable_peft:>18.2e}{peft_overhead_gb:>30.3f}")

    print(f"\n-> Looking only at the gradient + optimizer-state overhead -- the part")
    print(f"   that scales with how many parameters are TRAINABLE, not how many exist --")
    print(f"   the gap is exactly 1/{PEFT_TRAINABLE_FRACTION} = {1 / PEFT_TRAINABLE_FRACTION:.0f}x at every model size, a genuinely")
    print("   multi-order-of-magnitude difference. It just gets partly hidden in the")
    print("   TOTAL-memory figures above because both approaches still have to store")
    print("   the frozen base weights to run a forward pass at all.")


model_size_sweep_demo()

## 3. Multi-task deployment: N-টি full checkpoint বনাম 1-টি base + N-টি adapter

একটি 7B base model থেকে 5-টি আলাদা fine-tuned task serve করতে কত storage লাগে তা তুলনা করা হয়: প্রতিটি task-এর জন্য একটি করে পূর্ণ fp16 checkpoint, বনাম একটি shared base সাথে প্রতি task-এ একটি ছোট adapter। প্রতিটি নতুন task যোগ হলে ব্যবধান আরও বাড়ে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Multi-task deployment: N-টি full checkpoint বনাম 1-টি base + N-টি adapter
# ---------------------------------------------------------------------------

def multi_task_storage_demo():
    print("\n" + "=" * 78)
    print("3. STORING FINE-TUNED VARIANTS FOR MULTIPLE TASKS")
    print("=" * 78)

    n_total = 7e9  # Llama-2-7B scale
    n_trainable = n_total * PEFT_TRAINABLE_FRACTION
    num_tasks = 5

    full_checkpoint_gb = n_total * BYTES_WEIGHTS_FP16 / GB       # একটি fp16 checkpoint
    adapter_checkpoint_gb = n_trainable * BYTES_WEIGHTS_FP16 / GB  # একটি fp16 adapter

    full_ft_total_gb = num_tasks * full_checkpoint_gb
    peft_total_gb = full_checkpoint_gb + num_tasks * adapter_checkpoint_gb  # 1-টি shared base

    print(f"Base model: 7B params. Serving {num_tasks} independently fine-tuned tasks.\n")
    print(f"  one full fp16 checkpoint:        {full_checkpoint_gb:.2f} GB")
    print(f"  one fp16 adapter checkpoint:     {adapter_checkpoint_gb * 1024:.1f} MB "
          f"({PEFT_TRAINABLE_FRACTION * 100:.1f}% of the base model's size)\n")
    print(f"  Full fine-tuning: {num_tasks} independent full checkpoints "
          f"= {full_ft_total_gb:.1f} GB total")
    print(f"  PEFT: 1 shared base + {num_tasks} adapters "
          f"= {peft_total_gb:.2f} GB total")
    print(f"\n-> Serving {num_tasks} fine-tuned tasks from one base model needs "
          f"{full_ft_total_gb / peft_total_gb:.1f}x less storage")
    print("   with PEFT than with independently fully fine-tuned copies -- and the gap")
    print("   widens with every additional task, since each new full-FT task adds an")
    print(f"   entire {full_checkpoint_gb:.1f} GB checkpoint, while each new PEFT task adds only")
    print(f"   {adapter_checkpoint_gb * 1024:.1f} MB.")


multi_task_storage_demo()

## সবগুলো demo একসাথে

`main()` function-টি ওপরের তিনটি demo এক সাথে, ক্রমানুসারে চালায়। ওপরের cell-গুলোতে প্রতিটি demo ইতিমধ্যেই চলেছে, তাই নিচের call-টি comment করে রাখা হয়েছে — সবগুলো আবার একসাথে চালাতে চাইলে uncomment করুন।

In [ ]:
def main():
    per_parameter_breakdown_demo()
    model_size_sweep_demo()
    multi_task_storage_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন